# Fine-tune DINOv2 cho ảnh ngoài vườn

Chọn GPU T4. Tải `dinov2-improvement-source.zip` vào bộ nhớ phiên. Notebook dùng lại ảnh và weights của lượt trước nếu chúng còn trong runtime. Với runtime mới, tải thêm `dinov2-field-v2.zip` và `tomato-dinov2-results-20261111.zip`.

Rachin đã xem ở lượt trước nên chỉ dùng để phát triển. Lượt train chọn checkpoint bằng validation thực địa theo nguồn và recall âm. Tập kiểm thử mới chỉ được đưa vào sau khi checkpoint đã chọn.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import importlib.metadata
import json
import subprocess
import sys

try:
    version = importlib.metadata.version("transformers")
except importlib.metadata.PackageNotFoundError:
    version = None
if version != "4.57.6":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6"], check=True)

ROOT = Path("/content/tomato-dinov2-improve")
PRIOR = Path("/content/tomato-dinov2")
DATA = PRIOR / "data"
INITIAL = PRIOR / "run/model"
RUN = ROOT / "run"
ROOT.mkdir(exist_ok=True)

def unpack(package, target, prefix=None):
    if not package.is_file():
        raise FileNotFoundError(f"Upload {package.name} to Colab session storage")
    with ZipFile(package) as archive:
        members = [member for member in archive.infolist() if prefix is None or member.filename.startswith(prefix)]
        if not members:
            raise ValueError(f"Missing expected archive members in {package.name}")
        for member in members:
            if not (target / member.filename).resolve().is_relative_to(target.resolve()):
                raise ValueError(f"Unsafe archive path: {member.filename}")
        archive.extractall(target, members)

unpack(Path("/content/dinov2-improvement-source.zip"), ROOT)
if not (DATA / "images").is_dir():
    unpack(Path("/content/dinov2-field-v2.zip"), DATA)
if not (INITIAL / "model.safetensors").is_file():
    unpack(Path("/content/tomato-dinov2-results-20261111.zip"), PRIOR / "run", prefix="model/")
receipt = json.loads((ROOT / "development-receipt.json").read_text())
for name, field in (("development.csv", "development_sha256"), ("regression-controls.csv", "regression_controls_sha256")):
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == receipt[field]


In [ ]:
command = [
    sys.executable, "-u", "-m", "ai.dinov2_improve", "train",
    "--data", str(DATA), "--development", str(ROOT / "development.csv"),
    "--initial", str(INITIAL), "--output", str(RUN),
    "--epochs", "6", "--samples", "3072", "--batch", "24", "--size", "280",
]
with (ROOT / "train.log").open("a", encoding="utf-8") as log:
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)


In [ ]:
EXTERNAL = ROOT / "external"
test_sha256 = "09e46e7668f421a613ddc83b99c5d813da290b11c69ac406d59dd1ad786bb8a0"
size_control = ROOT / "prior-at-280"
size_control.mkdir(exist_ok=True)
for source in INITIAL.iterdir():
    if source.is_file() and source.name != "leaf-classifier.json":
        target = size_control / source.name
        if not target.exists():
            target.hardlink_to(source)
        elif not target.samefile(source):
            raise ValueError(f"Size-control weights differ: {target}")
same_size = json.loads((INITIAL / "leaf-classifier.json").read_text())
same_size["size"] = json.loads((RUN / "model/leaf-classifier.json").read_text())["size"]
(size_control / "leaf-classifier.json").write_text(json.dumps(same_size))
frozen = {"test_manifest_sha256": test_sha256, "weights": {}}
for label, model in (("prior", INITIAL), ("candidate", RUN / "model"), ("prior_size280", size_control)):
    frozen["weights"][label] = {}
    for path in sorted(model.iterdir()):
        if path.is_file():
            with path.open("rb") as stream:
                frozen["weights"][label][path.name] = hashlib.file_digest(stream, "sha256").hexdigest()
freeze_path = RUN / "frozen-models.json"
if freeze_path.exists() and json.loads(freeze_path.read_text()) != frozen:
    raise ValueError("Model or test manifest changed after freeze")
freeze_path.write_text(json.dumps(frozen, indent=2) + "\n")
unpack(Path("/content/dinov2-external-v1.zip"), EXTERNAL)
assert hashlib.sha256((EXTERNAL / "test.csv").read_bytes()).hexdigest() == test_sha256
for label, model in (("prior", INITIAL), ("candidate", RUN / "model")):
    for label_set, data_root, manifest in (
        ("external", EXTERNAL, EXTERNAL / "test.csv"),
        ("controls", DATA, ROOT / "regression-controls.csv"),
    ):
        subprocess.run(
            [sys.executable, "-u", "-m", "ai.dinov2_improve", "predict",
             "--data", str(data_root), "--manifest", str(manifest), "--model", str(model),
             "--predictions", str(RUN / f"{label_set}-{label}.csv")],
            cwd=ROOT, check=True,
        )
subprocess.run(
    [sys.executable, "-u", "-m", "ai.dinov2_improve", "predict",
     "--data", str(EXTERNAL), "--manifest", str(EXTERNAL / "test.csv"), "--model", str(size_control),
     "--predictions", str(RUN / "external-prior-size280.csv")],
    cwd=ROOT, check=True,
)
print("External and regression predictions completed.")


In [ ]:
from zipfile import ZIP_STORED

for name, include in (
    ("dinov2-improvement-model-20261111.zip", sorted((RUN / "model").iterdir())),
    ("dinov2-improvement-evidence-20261111.zip",
     sorted(path for path in RUN.iterdir() if path.suffix in (".csv", ".json")) + [ROOT / "train.log", ROOT / "development-receipt.json"]),
):
    package = Path("/content") / name
    with ZipFile(package, "w", ZIP_STORED) as archive:
        for path in include:
            archive.write(path, f"model/{path.name}" if path.parent == RUN / "model" else path.name)
    with package.open("rb") as stream:
        print(name, package.stat().st_size, hashlib.file_digest(stream, "sha256").hexdigest())
